# 1. Weekly LSTM Input Preparation

This notebook prepares the inputs for the final weekly modelling framework: a weekly HMM-informed LSTM classifier.

The notebook only prepares data. It does not train an HMM and it does not train an LSTM. The HMM notebook will later use `Weekly_Log_Return` only, and the later LSTM training notebook will use the weekly feature sequences prepared here.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 5

tickers = ["AAPL", "MSFT", "IBM"]
start_date = "2010-01-01"
end_date = "2026-01-01"

WEEKLY_OUTPUT_DIR = Path("outputs/weekly")
SEQUENCE_DIR = WEEKLY_OUTPUT_DIR / "sequences"

LOOKBACK_GRID = [4, 8, 12, 26]

WEEKLY_LSTM_FEATURE_COLS = [
    "Weekly_Log_Return",
    "Weekly_Open_Close_Log_Return",
    "Weekly_High_Low_Range",
    "Weekly_Volume_Change",
    "Rolling_Vol_4",
    "Rolling_Vol_12",
    "Rolling_Vol_26",
    "Momentum_4",
    "Momentum_12",
    "Momentum_26",
    "MA_Gap_4",
    "MA_Gap_12",
    "MA_Gap_26",
    "Drawdown_12",
    "Drawdown_26",
]

WEEKLY_HMM_INPUT_COLS_LATER = ["Weekly_Log_Return"]


## 2. Load daily OHLCV data

This uses the same raw Yahoo Finance download and long-table conversion pattern as the original daily preprocessing notebook. The resulting daily dataframe has one row per ticker-date and the raw OHLCV columns needed for weekly aggregation.


In [2]:
def yfinance_to_long(raw: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    """Convert yfinance output into a long OHLCV table."""
    frames = []

    if isinstance(raw.columns, pd.MultiIndex):
        level_0 = raw.columns.get_level_values(0)
        level_1 = raw.columns.get_level_values(1)

        for ticker in tickers:
            if ticker in level_0:
                tmp = raw[ticker].copy()
            elif ticker in level_1:
                tmp = raw.xs(ticker, axis=1, level=1).copy()
            else:
                raise ValueError(f"Ticker {ticker} was not found in yfinance output.")

            tmp = tmp.reset_index()
            tmp["Ticker"] = ticker
            frames.append(tmp)
    else:
        tmp = raw.reset_index().copy()
        tmp["Ticker"] = tickers[0]
        frames.append(tmp)

    long_df = pd.concat(frames, ignore_index=True)

    required_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]
    missing_cols = [col for col in required_cols if col not in long_df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    long_df = long_df[required_cols].copy()
    long_df["Date"] = pd.to_datetime(long_df["Date"])

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        long_df[col] = pd.to_numeric(long_df[col], errors="coerce")

    return long_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)


raw_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    group_by="ticker",
    progress=True,
)

raw_df = yfinance_to_long(raw_data, tickers)

required_raw_cols = ["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
missing_raw_cols = [col for col in required_raw_cols if col not in raw_df.columns]
if missing_raw_cols:
    raise ValueError(f"Missing raw columns: {missing_raw_cols}")

raw_df["Date"] = pd.to_datetime(raw_df["Date"])
raw_df = raw_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Raw daily OHLCV shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.groupby("Ticker").size())
display(raw_df[["Date"]].agg(["min", "max"]))



[                       0%                       ]


[**********************67%*******                ]  2 of 3 completed


[*********************100%***********************]  3 of 3 completed

Raw daily OHLCV shape: (12072, 7)


Price,Date,Ticker,Open,High,Low,Close,Volume
0,2010-01-04,AAPL,6.389118,6.421149,6.357686,6.406481,493729600
1,2010-01-05,AAPL,6.424143,6.453779,6.383729,6.417557,601904800
2,2010-01-06,AAPL,6.417558,6.443003,6.308892,6.315478,552160000
3,2010-01-07,AAPL,6.338825,6.346309,6.257999,6.303800,477131200
4,2010-01-08,AAPL,6.295420,6.346310,6.258301,6.345711,447610800


Ticker
AAPL    4024
IBM     4024
MSFT    4024
dtype: int64

Price,Date
min,2010-01-04
max,2025-12-31


## 3. Resample to weekly OHLCV

Weekly rows use Friday as the week-ending date. For each ticker-week, `Open` is the first open, `High` is the maximum high, `Low` is the minimum low, `Close` is the last close, and `Volume` is summed.


In [3]:
def make_weekly_ohlcv(group: pd.DataFrame) -> pd.DataFrame:
    ticker = group["Ticker"].iloc[0]

    g = group.sort_values("Date").copy()
    g["Date"] = pd.to_datetime(g["Date"])
    g = g.set_index("Date")

    weekly = (
        g.resample("W-FRI")
        .agg({
            "Open": "first",
            "High": "max",
            "Low": "min",
            "Close": "last",
            "Volume": "sum",
        })
        .dropna(subset=["Open", "High", "Low", "Close"])
        .reset_index()
    )

    weekly["Ticker"] = ticker
    return weekly[["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]]


weekly_ohlcv_df = pd.concat(
    [make_weekly_ohlcv(group) for _, group in raw_df.groupby("Ticker", sort=True)],
    ignore_index=True,
)
weekly_ohlcv_df = weekly_ohlcv_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

display(weekly_ohlcv_df.head())
display(weekly_ohlcv_df.groupby("Ticker").size())
display(weekly_ohlcv_df[["Date"]].agg(["min", "max"]))


Price,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2010-01-08,6.389118,6.453779,6.257999,6.345711,2572536400
1,AAPL,2010-01-15,6.370257,6.376244,6.109819,6.164603,2689542800
2,AAPL,2010-01-22,6.236448,6.452581,5.902069,5.919730,2832082400
3,AAPL,2010-01-29,6.062223,6.397500,5.695214,5.749397,7074832800
4,AAPL,2010-02-05,5.758677,5.993073,5.713176,5.851179,3671505600


Ticker
AAPL    835
IBM     835
MSFT    835
dtype: int64

Price,Date
min,2010-01-08
max,2026-01-02


## 4. Weekly feature engineering

Features are created separately per ticker, using only current and past weekly information. No HMM features are included here; `HMM_Bullish_Prob_Next` will be added later by the weekly HMM notebook.


In [4]:
def add_weekly_features(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values("Date").copy()

    g["Log_Close"] = np.log(g["Close"])
    g["Weekly_Log_Return"] = g["Log_Close"].diff()

    g["Weekly_Open_Close_Log_Return"] = np.log(g["Close"] / g["Open"])
    g["Weekly_High_Low_Range"] = (g["High"] - g["Low"]) / g["Close"]

    g["Log_Weekly_Volume"] = np.log(g["Volume"].replace(0, np.nan))
    g["Weekly_Volume_Change"] = g["Log_Weekly_Volume"].diff()

    for window in [4, 12, 26]:
        g[f"Rolling_Vol_{window}"] = (
            g["Weekly_Log_Return"]
            .rolling(window)
            .std()
        )

        g[f"Momentum_{window}"] = (
            g["Weekly_Log_Return"]
            .rolling(window)
            .sum()
        )

        g[f"MA_Gap_{window}"] = (
            g["Close"] / g["Close"].rolling(window).mean() - 1
        )

    g["Drawdown_12"] = (
        g["Close"] / g["Close"].rolling(12).max() - 1
    )

    g["Drawdown_26"] = (
        g["Close"] / g["Close"].rolling(26).max() - 1
    )

    return g


weekly_features_df = pd.concat(
    [add_weekly_features(group) for _, group in weekly_ohlcv_df.groupby("Ticker", sort=True)],
    ignore_index=True,
)
weekly_features_df = weekly_features_df.replace([np.inf, -np.inf], np.nan)
weekly_features_df = weekly_features_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

missing_feature_cols = [col for col in WEEKLY_LSTM_FEATURE_COLS if col not in weekly_features_df.columns]
if missing_feature_cols:
    raise ValueError(f"Missing weekly feature columns: {missing_feature_cols}")

print("Weekly feature columns:")
print(WEEKLY_LSTM_FEATURE_COLS)
display(weekly_features_df[["Ticker", "Date", "Close"] + WEEKLY_LSTM_FEATURE_COLS].head(30))


Weekly feature columns:
['Weekly_Log_Return', 'Weekly_Open_Close_Log_Return', 'Weekly_High_Low_Range', 'Weekly_Volume_Change', 'Rolling_Vol_4', 'Rolling_Vol_12', 'Rolling_Vol_26', 'Momentum_4', 'Momentum_12', 'Momentum_26', 'MA_Gap_4', 'MA_Gap_12', 'MA_Gap_26', 'Drawdown_12', 'Drawdown_26']


Price,Ticker,Date,Close,Weekly_Log_Return,Weekly_Open_Close_Log_Return,Weekly_High_Low_Range,Weekly_Volume_Change,Rolling_Vol_4,Rolling_Vol_12,Rolling_Vol_26,Momentum_4,Momentum_12,Momentum_26,MA_Gap_4,MA_Gap_12,MA_Gap_26,Drawdown_12,Drawdown_26
0,AAPL,2010-01-08,6.345711,NaN,-0.006817,0.030852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,2010-01-15,6.164603,-0.028955,-0.032816,0.043219,0.044479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL,2010-01-22,5.919730,-0.040533,-0.052120,0.092996,0.051641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL,2010-01-29,5.749397,-0.029196,-0.052982,0.122149,0.915532,NaN,NaN,NaN,NaN,NaN,NaN,-0.048878,NaN,NaN,NaN,NaN
4,AAPL,2010-02-05,5.851179,0.017548,0.015935,0.047836,-0.655942,0.025793,NaN,NaN,-0.081136,NaN,NaN,-0.011830,NaN,NaN,NaN,NaN
5,AAPL,2010-02-12,5.998461,0.024860,0.023684,0.038128,-0.312052,0.032836,NaN,NaN,-0.027321,NaN,NaN,0.020200,NaN,NaN,NaN,NaN
6,AAPL,2010-02-19,6.037076,0.006417,-0.001338,0.017107,-0.390576,0.023967,NaN,NaN,0.019629,NaN,NaN,0.021670,NaN,NaN,NaN,NaN
7,AAPL,2010-02-26,6.125386,0.014522,0.011205,0.046232,0.357081,0.007633,NaN,NaN,0.063347,NaN,NaN,0.020383,NaN,NaN,NaN,NaN
8,AAPL,2010-03-05,6.554361,0.067689,0.062182,0.065084,0.058129,0.027276,NaN,NaN,0.113488,NaN,NaN,0.060779,NaN,NaN,NaN,NaN
9,AAPL,2010-03-12,6.783369,0.034343,0.029514,0.043425,0.005083,0.027282,NaN,NaN,0.122971,NaN,NaN,0.064050,NaN,NaN,NaN,NaN


## 5. Next-week target construction

At week `t`, the LSTM inputs use information up to week `t`. The target is the sign of the log return in week `t+1`.


In [5]:
def add_next_week_target(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values("Date").copy()

    g["Target_Date"] = g["Date"].shift(-1)
    g["Next_Week_Log_Return"] = g["Weekly_Log_Return"].shift(-1)

    g["Target_State_Binary"] = np.where(g["Next_Week_Log_Return"] > 0, 1, 0)
    g.loc[g["Next_Week_Log_Return"].isna(), "Target_State_Binary"] = np.nan

    g["Target_State"] = np.where(
        g["Next_Week_Log_Return"] > 0,
        "Bullish",
        "Bearish",
    )
    g.loc[g["Next_Week_Log_Return"].isna(), "Target_State"] = np.nan

    return g


weekly_target_df = pd.concat(
    [add_next_week_target(group) for _, group in weekly_features_df.groupby("Ticker", sort=True)],
    ignore_index=True,
)
weekly_target_df = weekly_target_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

required_model_cols = [
    "Ticker",
    "Date",
    "Target_Date",
    "Weekly_Log_Return",
    "Next_Week_Log_Return",
    "Target_State_Binary",
] + WEEKLY_LSTM_FEATURE_COLS

required_model_cols = list(dict.fromkeys(required_model_cols))

model_df_weekly = weekly_target_df.dropna(subset=required_model_cols).copy()
model_df_weekly["Date"] = pd.to_datetime(model_df_weekly["Date"])
model_df_weekly["Target_Date"] = pd.to_datetime(model_df_weekly["Target_Date"])
model_df_weekly["Target_State_Binary"] = model_df_weekly["Target_State_Binary"].astype(int)
model_df_weekly = model_df_weekly.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Weekly modelling dataframe shape:", model_df_weekly.shape)
display(model_df_weekly[["Ticker", "Date", "Target_Date", "Weekly_Log_Return", "Next_Week_Log_Return", "Target_State_Binary", "Target_State"]].head())
display(model_df_weekly.groupby("Ticker").size())


Weekly modelling dataframe shape: (2424, 28)


Price,Ticker,Date,Target_Date,Weekly_Log_Return,Next_Week_Log_Return,Target_State_Binary,Target_State
0,AAPL,2010-07-09,2010-07-16,0.050073,-0.038158,0,Bearish
1,AAPL,2010-07-16,2010-07-23,-0.038158,0.039390,1,Bullish
2,AAPL,2010-07-23,2010-07-30,0.039390,-0.010402,0,Bearish
3,AAPL,2010-07-30,2010-08-06,-0.010402,0.010979,1,Bullish
4,AAPL,2010-08-06,2010-08-13,0.010979,-0.043174,0,Bearish


Ticker
AAPL    808
IBM     808
MSFT    808
dtype: int64

## 6. Chronological train/validation/test split

Splits are assigned separately per ticker using `Target_Date`, because the prediction target belongs to the next week. The first 70% of target weeks are training, the next 15% are validation, and the final 15% are test.


In [6]:
def assign_chronological_split(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values("Target_Date").copy()
    n = len(g)

    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    split_values = np.empty(n, dtype=object)
    split_values[:train_end] = "train"
    split_values[train_end:val_end] = "validation"
    split_values[val_end:] = "test"

    g["Split"] = split_values
    return g


model_df_weekly = pd.concat(
    [assign_chronological_split(group) for _, group in model_df_weekly.groupby("Ticker", sort=True)],
    ignore_index=True,
)
model_df_weekly = model_df_weekly.sort_values(["Ticker", "Date"]).reset_index(drop=True)

display(model_df_weekly.groupby(["Ticker", "Split"]).size().unstack(fill_value=0))
display(model_df_weekly.groupby(["Ticker", "Split", "Target_State"]).size().unstack(fill_value=0))
split_boundary_summary = (
    model_df_weekly
    .groupby(["Ticker", "Split"])
    .agg(
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        target_start=("Target_Date", "min"),
        target_end=("Target_Date", "max"),
        rows=("Date", "count"),
    )
)
display(split_boundary_summary)

split_boundary_rows = []
for ticker, group in model_df_weekly.groupby("Ticker", sort=True):
    bounds = group.groupby("Split")["Target_Date"].agg(["min", "max"])
    missing_splits = [split for split in ["train", "validation", "test"] if split not in bounds.index]
    if missing_splits:
        raise ValueError(f"Ticker {ticker} is missing split(s): {missing_splits}")

    train_max = bounds.loc["train", "max"]
    validation_min = bounds.loc["validation", "min"]
    validation_max = bounds.loc["validation", "max"]
    test_min = bounds.loc["test", "min"]

    train_before_validation = train_max < validation_min
    validation_before_test = validation_max < test_min

    if not train_before_validation:
        raise ValueError(
            f"Chronological split boundary failed for {ticker}: "
            f"train max Target_Date {train_max} is not before validation min Target_Date {validation_min}."
        )
    if not validation_before_test:
        raise ValueError(
            f"Chronological split boundary failed for {ticker}: "
            f"validation max Target_Date {validation_max} is not before test min Target_Date {test_min}."
        )

    split_boundary_rows.append(
        {
            "Ticker": ticker,
            "train_max_Target_Date": train_max,
            "validation_min_Target_Date": validation_min,
            "validation_max_Target_Date": validation_max,
            "test_min_Target_Date": test_min,
            "train_max_before_validation_min": train_before_validation,
            "validation_max_before_test_min": validation_before_test,
        }
    )

split_boundary_check_df = pd.DataFrame(split_boundary_rows)
display(split_boundary_check_df)
print("Chronological split boundary checks passed for every ticker.")


Split,test,train,validation
Ticker,,,
AAPL,122,565,121
IBM,122,565,121
MSFT,122,565,121


Target_State       Bearish  Bullish
Ticker Split                       
AAPL   test             55       67
       train           235      330
       validation       55       66
IBM    test             48       74
       train           269      296
       validation       56       65
MSFT   test             53       69
       train           235      330
       validation       57       64

start_date   end_date target_start target_end  rows
Ticker Split                                                         
AAPL   test       2023-09-01 2025-12-26   2023-09-08 2026-01-02   122
       train      2010-07-09 2021-04-30   2010-07-16 2021-05-07   565
       validation 2021-05-07 2023-08-25   2021-05-14 2023-09-01   121
IBM    test       2023-09-01 2025-12-26   2023-09-08 2026-01-02   122
       train      2010-07-09 2021-04-30   2010-07-16 2021-05-07   565
       validation 2021-05-07 2023-08-25   2021-05-14 2023-09-01   121
MSFT   test       2023-09-01 2025-12-26   2023-09-08 2026-01-02   122
       train      2010-07-09 2021-04-30   2010-07-16 2021-05-07   565
       validation 2021-05-07 2023-08-25   2021-05-14 2023-09-01   121

,Ticker,train_max_Target_Date,validation_min_Target_Date,validation_max_Target_Date,test_min_Target_Date,train_max_before_validation_min,validation_max_before_test_min
0,AAPL,2021-05-07,2021-05-14,2023-09-01,2023-09-08,True,True
1,IBM,2021-05-07,2021-05-14,2023-09-01,2023-09-08,True,True
2,MSFT,2021-05-07,2021-05-14,2023-09-01,2023-09-08,True,True


Chronological split boundary checks passed for every ticker.


## 7. Scaling without leakage

A separate `StandardScaler` is fitted for each ticker using training rows only. The fitted ticker-specific scaler is then applied to that ticker's training, validation, and test rows. Only `WEEKLY_LSTM_FEATURE_COLS` are scaled.


In [7]:
model_df_weekly_scaled = model_df_weekly.copy()
scaler_metadata_rows = []

for ticker, group in model_df_weekly.groupby("Ticker", sort=True):
    train_mask = group["Split"] == "train"
    if not train_mask.any():
        raise ValueError(f"Ticker {ticker} has no training rows for scaler fitting.")

    scaler = StandardScaler()
    scaler.fit(group.loc[train_mask, WEEKLY_LSTM_FEATURE_COLS])

    transformed = scaler.transform(group[WEEKLY_LSTM_FEATURE_COLS])
    model_df_weekly_scaled.loc[group.index, WEEKLY_LSTM_FEATURE_COLS] = transformed

    for feature, mean, scale in zip(WEEKLY_LSTM_FEATURE_COLS, scaler.mean_, scaler.scale_):
        scaler_metadata_rows.append(
            {
                "Ticker": ticker,
                "Feature": feature,
                "Train_Mean": mean,
                "Train_Scale": scale,
            }
        )

scaler_metadata_df = pd.DataFrame(scaler_metadata_rows)

non_feature_cols = [col for col in model_df_weekly.columns if col not in WEEKLY_LSTM_FEATURE_COLS]
preserved_non_feature_cols = [col for col in ["Ticker", "Date", "Target_Date", "Split", "Next_Week_Log_Return", "Target_State_Binary", "Target_State"] if col in model_df_weekly_scaled.columns]

print("Preserved non-feature columns:", preserved_non_feature_cols)
display(model_df_weekly_scaled[preserved_non_feature_cols + WEEKLY_LSTM_FEATURE_COLS[:5]].head())
display(scaler_metadata_df.head())
display(model_df_weekly_scaled.loc[model_df_weekly_scaled["Split"] == "train"].groupby("Ticker")[WEEKLY_LSTM_FEATURE_COLS].agg(["mean", "std"]).round(4))


Preserved non-feature columns: ['Ticker', 'Date', 'Target_Date', 'Split', 'Next_Week_Log_Return', 'Target_State_Binary', 'Target_State']


Price,Ticker,Date,Target_Date,Split,Next_Week_Log_Return,Target_State_Binary,Target_State,Weekly_Log_Return,Weekly_Open_Close_Log_Return,Weekly_High_Low_Range,Weekly_Volume_Change,Rolling_Vol_4
0,AAPL,2010-07-09,2010-07-16,train,-0.038158,0,Bearish,1.163815,0.792702,0.480823,-1.559593,1.945813
1,AAPL,2010-07-16,2010-07-23,train,0.039390,1,Bullish,-1.116691,-1.044977,0.380697,1.755855,1.006544
2,AAPL,2010-07-23,2010-07-30,train,-0.010402,0,Bearish,0.887679,0.947492,1.701277,0.032857,1.433856
3,AAPL,2010-07-30,2010-08-06,train,0.010979,1,Bullish,-0.399294,-0.412203,-0.290485,-1.551900,0.393701
4,AAPL,2010-08-06,2010-08-13,train,-0.043174,0,Bearish,0.153362,-0.160134,-0.923609,-0.776437,-0.072647


,Ticker,Feature,Train_Mean,Train_Scale
0,AAPL,Weekly_Log_Return,0.005046,0.038689
1,AAPL,Weekly_Open_Close_Log_Return,0.004556,0.036849
2,AAPL,Weekly_High_Low_Range,0.051157,0.027706
3,AAPL,Weekly_Volume_Change,-0.003750,0.341581
4,AAPL,Rolling_Vol_4,0.034213,0.018896


Price  Weekly_Log_Return         Weekly_Open_Close_Log_Return         Weekly_High_Low_Range         Weekly_Volume_Change         Rolling_Vol_4          \
                    mean     std                         mean     std                  mean     std                 mean     std          mean     std   
Ticker                                                                                                                                                   
AAPL                 0.0  1.0009                          0.0  1.0009                   0.0  1.0009                 -0.0  1.0009          -0.0  1.0009   
IBM                  0.0  1.0009                         -0.0  1.0009                  -0.0  1.0009                  0.0  1.0009           0.0  1.0009   
MSFT                -0.0  1.0009                          0.0  1.0009                   0.0  1.0009                 -0.0  1.0009           0.0  1.0009   

Price  Rolling_Vol_12         Rolling_Vol_26         Momentum_4         Momentum_12         Momentum_26         MA_Gap_4         MA_Gap_12         MA_Gap_26  \
                 mean     std           mean     std       mean     std        mean     std        mean     std     mean     std      mean     std      mean   
Ticker                                                                                                                                                         
AAPL             -0.0  1.0009            0.0  1.0009       -0.0  1.0009         0.0  1.0009         0.0  1.0009     -0.0  1.0009      -0.0  1.0009       0.0   
IBM              -0.0  1.0009           -0.0  1.0009       -0.0  1.0009         0.0  1.0009        -0.0  1.0009     -0.0  1.0009      -0.0  1.0009      -0.0   
MSFT             -0.0  1.0009            0.0  1.0009        0.0  1.0009         0.0  1.0009         0.0  1.0009      0.0  1.0009      -0.0  1.0009      -0.0   

Price          Drawdown_12         Drawdown_26          
           std        mean     std        mean     std  
Ticker                                                  
AAPL    1.0009         0.0  1.0009        -0.0  1.0009  
IBM     1.0009        -0.0  1.0009         0.0  1.0009  
MSFT    1.0009        -0.0  1.0009        -0.0  1.0009

## 8. Sequence creation for multiple lookback windows

Sequences are built separately per ticker. For a target row at position `i`, the input is rows `i - lookback + 1 : i`, including the current week. The target is `Target_State_Binary` at row `i`, which already represents the next-week direction.

Validation and test target rows may use earlier rows from a previous split as input history, because those weeks are in the past at prediction time.


In [8]:
def create_weekly_lstm_sequences(
    df: pd.DataFrame,
    feature_cols: list[str],
    lookback: int,
    target_col: str = "Target_State_Binary",
):
    X_by_split = {"train": [], "validation": [], "test": []}
    y_by_split = {"train": [], "validation": [], "test": []}
    sequence_info_rows = []

    df = df.sort_values(["Ticker", "Date"]).copy()

    for ticker, group in df.groupby("Ticker", sort=True):
        g = group.sort_values("Date").reset_index(drop=True)

        feature_values = g[feature_cols].to_numpy(dtype=np.float32)
        targets = g[target_col].to_numpy(dtype=np.int64)
        splits = g["Split"].to_numpy()
        dates = g["Date"].to_numpy()
        target_dates = g["Target_Date"].to_numpy()

        for i in range(lookback - 1, len(g)):
            split = splits[i]
            if split not in X_by_split:
                raise ValueError(f"Unexpected split value: {split}")

            X_by_split[split].append(feature_values[i - lookback + 1:i + 1])
            y_by_split[split].append(targets[i])
            sequence_info_rows.append(
                {
                    "Ticker": ticker,
                    "Lookback": lookback,
                    "Date": dates[i],
                    "Target_Date": target_dates[i],
                    "Split": split,
                    "Target_State_Binary": targets[i],
                }
            )

    def stack_X(split: str) -> np.ndarray:
        if X_by_split[split]:
            return np.asarray(X_by_split[split], dtype=np.float32)
        return np.empty((0, lookback, len(feature_cols)), dtype=np.float32)

    def stack_y(split: str) -> np.ndarray:
        if y_by_split[split]:
            return np.asarray(y_by_split[split], dtype=np.int64)
        return np.empty((0,), dtype=np.int64)

    sequence_info_df = pd.DataFrame(sequence_info_rows)

    return (
        stack_X("train"), stack_y("train"),
        stack_X("validation"), stack_y("validation"),
        stack_X("test"), stack_y("test"),
        sequence_info_df,
    )


weekly_sequence_arrays = {}
sequence_metadata_frames = []
sequence_shape_rows = []

for ticker in tickers:
    ticker_df = model_df_weekly_scaled.loc[model_df_weekly_scaled["Ticker"] == ticker].copy()

    for lookback in LOOKBACK_GRID:
        X_train, y_train, X_val, y_val, X_test, y_test, sequence_info_df = create_weekly_lstm_sequences(
            ticker_df,
            WEEKLY_LSTM_FEATURE_COLS,
            lookback,
        )

        weekly_sequence_arrays[(ticker, lookback)] = {
            "X_train": X_train,
            "y_train": y_train,
            "X_val": X_val,
            "y_val": y_val,
            "X_test": X_test,
            "y_test": y_test,
        }
        sequence_metadata_frames.append(sequence_info_df)

        sequence_shape_rows.append(
            {
                "Ticker": ticker,
                "Lookback": lookback,
                "X_train_shape": X_train.shape,
                "y_train_shape": y_train.shape,
                "X_val_shape": X_val.shape,
                "y_val_shape": y_val.shape,
                "X_test_shape": X_test.shape,
                "y_test_shape": y_test.shape,
            }
        )

weekly_sequence_metadata_df = pd.concat(sequence_metadata_frames, ignore_index=True)
sequence_shape_summary_df = pd.DataFrame(sequence_shape_rows)

display(sequence_shape_summary_df)
display(weekly_sequence_metadata_df.head())
display(weekly_sequence_metadata_df.groupby(["Ticker", "Lookback", "Split"]).size().unstack(fill_value=0))


,Ticker,Lookback,X_train_shape,y_train_shape,X_val_shape,y_val_shape,X_test_shape,y_test_shape
0,AAPL,4,"(562, 4, 15)","(562,)","(121, 4, 15)","(121,)","(122, 4, 15)","(122,)"
1,AAPL,8,"(558, 8, 15)","(558,)","(121, 8, 15)","(121,)","(122, 8, 15)","(122,)"
2,AAPL,12,"(554, 12, 15)","(554,)","(121, 12, 15)","(121,)","(122, 12, 15)","(122,)"
3,AAPL,26,"(540, 26, 15)","(540,)","(121, 26, 15)","(121,)","(122, 26, 15)","(122,)"
4,MSFT,4,"(562, 4, 15)","(562,)","(121, 4, 15)","(121,)","(122, 4, 15)","(122,)"
5,MSFT,8,"(558, 8, 15)","(558,)","(121, 8, 15)","(121,)","(122, 8, 15)","(122,)"
6,MSFT,12,"(554, 12, 15)","(554,)","(121, 12, 15)","(121,)","(122, 12, 15)","(122,)"
7,MSFT,26,"(540, 26, 15)","(540,)","(121, 26, 15)","(121,)","(122, 26, 15)","(122,)"
8,IBM,4,"(562, 4, 15)","(562,)","(121, 4, 15)","(121,)","(122, 4, 15)","(122,)"
9,IBM,8,"(558, 8, 15)","(558,)","(121, 8, 15)","(121,)","(122, 8, 15)","(122,)"


,Ticker,Lookback,Date,Target_Date,Split,Target_State_Binary
0,AAPL,4,2010-07-30,2010-08-06,train,1
1,AAPL,4,2010-08-06,2010-08-13,train,0
2,AAPL,4,2010-08-13,2010-08-20,train,1
3,AAPL,4,2010-08-20,2010-08-27,train,0
4,AAPL,4,2010-08-27,2010-09-03,train,1


Split            test  train  validation
Ticker Lookback                         
AAPL   4          122    562         121
       8          122    558         121
       12         122    554         121
       26         122    540         121
IBM    4          122    562         121
       8          122    558         121
       12         122    554         121
       26         122    540         121
MSFT   4          122    562         121
       8          122    558         121
       12         122    554         121
       26         122    540         121

## 9. Save weekly outputs

All weekly preparation outputs are saved under `outputs/weekly/`. Sequence arrays are saved per ticker and lookback so the later LSTM training notebook can load exactly the window it is testing.


In [9]:
WEEKLY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

required_saved_cols = [
    "Ticker",
    "Date",
    "Target_Date",
    "Split",
    "Weekly_Log_Return",
    "Next_Week_Log_Return",
    "Target_State_Binary",
    "Target_State",
]

missing_saved_cols = [col for col in required_saved_cols if col not in model_df_weekly.columns]
if missing_saved_cols:
    raise ValueError(f"Missing required saved columns: {missing_saved_cols}")

weekly_ohlcv_path = WEEKLY_OUTPUT_DIR / "weekly_ohlcv_df.parquet"
weekly_ohlcv_csv_path = WEEKLY_OUTPUT_DIR / "weekly_ohlcv_df.csv"
weekly_features_path = WEEKLY_OUTPUT_DIR / "weekly_features_df.parquet"
weekly_features_csv_path = WEEKLY_OUTPUT_DIR / "weekly_features_df.csv"
model_weekly_path = WEEKLY_OUTPUT_DIR / "model_df_weekly.parquet"
model_weekly_csv_path = WEEKLY_OUTPUT_DIR / "model_df_weekly.csv"
scaled_weekly_path = WEEKLY_OUTPUT_DIR / "model_df_weekly_scaled.parquet"
scaled_weekly_csv_path = WEEKLY_OUTPUT_DIR / "model_df_weekly_scaled.csv"
scaler_metadata_path = WEEKLY_OUTPUT_DIR / "weekly_scaler_metadata.csv"
sequence_metadata_path = WEEKLY_OUTPUT_DIR / "weekly_sequence_metadata.csv"
sequence_shape_summary_path = WEEKLY_OUTPUT_DIR / "weekly_sequence_shape_summary.csv"

weekly_ohlcv_df.to_parquet(weekly_ohlcv_path, index=False)
weekly_ohlcv_df.to_csv(weekly_ohlcv_csv_path, index=False)

weekly_features_df.to_parquet(weekly_features_path, index=False)
weekly_features_df.to_csv(weekly_features_csv_path, index=False)

model_df_weekly.to_parquet(model_weekly_path, index=False)
model_df_weekly.to_csv(model_weekly_csv_path, index=False)

model_df_weekly_scaled.to_parquet(scaled_weekly_path, index=False)
model_df_weekly_scaled.to_csv(scaled_weekly_csv_path, index=False)

scaler_metadata_df.to_csv(scaler_metadata_path, index=False)
weekly_sequence_metadata_df.to_csv(sequence_metadata_path, index=False)
sequence_shape_summary_df.to_csv(sequence_shape_summary_path, index=False)

saved_sequence_paths = []
for (ticker, lookback), arrays in weekly_sequence_arrays.items():
    sequence_path = SEQUENCE_DIR / f"{ticker}_lookback_{lookback}.npz"
    np.savez_compressed(
        sequence_path,
        X_train=arrays["X_train"],
        y_train=arrays["y_train"],
        X_val=arrays["X_val"],
        y_val=arrays["y_val"],
        X_test=arrays["X_test"],
        y_test=arrays["y_test"],
        feature_cols=np.asarray(WEEKLY_LSTM_FEATURE_COLS),
        lookback=np.asarray(lookback),
        ticker=np.asarray(ticker),
    )
    saved_sequence_paths.append(sequence_path)

saved_output_paths = [
    weekly_ohlcv_path,
    weekly_ohlcv_csv_path,
    weekly_features_path,
    weekly_features_csv_path,
    model_weekly_path,
    model_weekly_csv_path,
    scaled_weekly_path,
    scaled_weekly_csv_path,
    scaler_metadata_path,
    sequence_metadata_path,
    sequence_shape_summary_path,
] + saved_sequence_paths

for output_path in saved_output_paths:
    if not output_path.exists():
        raise FileNotFoundError(f"Expected output was not saved: {output_path}")

print("Saved weekly outputs:")
for output_path in saved_output_paths:
    print(output_path)


Saved weekly outputs:
outputs\weekly\weekly_ohlcv_df.parquet
outputs\weekly\weekly_ohlcv_df.csv
outputs\weekly\weekly_features_df.parquet
outputs\weekly\weekly_features_df.csv
outputs\weekly\model_df_weekly.parquet
outputs\weekly\model_df_weekly.csv
outputs\weekly\model_df_weekly_scaled.parquet
outputs\weekly\model_df_weekly_scaled.csv
outputs\weekly\weekly_scaler_metadata.csv
outputs\weekly\weekly_sequence_metadata.csv
outputs\weekly\weekly_sequence_shape_summary.csv
outputs\weekly\sequences\AAPL_lookback_4.npz
outputs\weekly\sequences\AAPL_lookback_8.npz
outputs\weekly\sequences\AAPL_lookback_12.npz
outputs\weekly\sequences\AAPL_lookback_26.npz
outputs\weekly\sequences\MSFT_lookback_4.npz
outputs\weekly\sequences\MSFT_lookback_8.npz
outputs\weekly\sequences\MSFT_lookback_12.npz
outputs\weekly\sequences\MSFT_lookback_26.npz
outputs\weekly\sequences\IBM_lookback_4.npz
outputs\weekly\sequences\IBM_lookback_8.npz
outputs\weekly\sequences\IBM_lookback_12.npz
outputs\weekly\sequences\IBM_l

## 10. Final sanity checks

These checks confirm the weekly preparation is classification-focused, uses binary next-week targets, has no required HMM columns yet, and saved all expected weekly files.


In [10]:
print("weekly_ohlcv_df shape:", weekly_ohlcv_df.shape)
print("model_df_weekly shape:", model_df_weekly.shape)

print("\nRows by ticker and split:")
display(model_df_weekly.groupby(["Ticker", "Split"]).size().unstack(fill_value=0))

print("\nClass balance by ticker and split:")
display(model_df_weekly.groupby(["Ticker", "Split", "Target_State"]).size().unstack(fill_value=0))

print("\nFinal WEEKLY_LSTM_FEATURE_COLS:")
print(WEEKLY_LSTM_FEATURE_COLS)

print("\nMissing values in feature columns:")
missing_feature_values = model_df_weekly[WEEKLY_LSTM_FEATURE_COLS].isna().sum()
display(missing_feature_values)
if missing_feature_values.sum() != 0:
    raise ValueError("Feature columns contain missing values after modelling-row filtering.")

print("\nSequence shapes by ticker and lookback:")
for row in sequence_shape_summary_df.itertuples(index=False):
    print(
        f"{row.Ticker} lookback={row.Lookback}: "
        f"X_train={row.X_train_shape}, y_train={row.y_train_shape}, "
        f"X_val={row.X_val_shape}, y_val={row.y_val_shape}, "
        f"X_test={row.X_test_shape}, y_test={row.y_test_shape}"
    )

target_values = set(model_df_weekly["Target_State_Binary"].unique())
sequence_target_values = set(weekly_sequence_metadata_df["Target_State_Binary"].unique())
if not target_values.issubset({0, 1}):
    raise ValueError(f"Model dataframe targets are not binary 0/1: {target_values}")
if not sequence_target_values.issubset({0, 1}):
    raise ValueError(f"Sequence targets are not binary 0/1: {sequence_target_values}")
print("\nAll dataframe and sequence targets are binary 0/1.")

required_sequence_files = [
    SEQUENCE_DIR / f"{ticker}_lookback_{lookback}.npz"
    for ticker in tickers
    for lookback in LOOKBACK_GRID
]
required_files = [
    WEEKLY_OUTPUT_DIR / "model_df_weekly.parquet",
    WEEKLY_OUTPUT_DIR / "model_df_weekly.csv",
    WEEKLY_OUTPUT_DIR / "model_df_weekly_scaled.parquet",
    WEEKLY_OUTPUT_DIR / "weekly_sequence_metadata.csv",
] + required_sequence_files

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing saved files: {missing_files}")
print("All required weekly output files exist.")

hmm_feature_cols_present = [col for col in WEEKLY_LSTM_FEATURE_COLS if col.startswith("HMM")]
hmm_columns_present = [col for col in model_df_weekly.columns if col.startswith("HMM")]
if hmm_feature_cols_present:
    raise ValueError(f"HMM columns should not be LSTM features yet: {hmm_feature_cols_present}")
print("No HMM columns are required yet.")
print("HMM input for the later weekly HMM notebook:", WEEKLY_HMM_INPUT_COLS_LATER)
print("HMM columns currently present in prepared dataframe:", hmm_columns_present)


weekly_ohlcv_df shape: (2505, 7)
model_df_weekly shape: (2424, 29)

Rows by ticker and split:


Split,test,train,validation
Ticker,,,
AAPL,122,565,121
IBM,122,565,121
MSFT,122,565,121



Class balance by ticker and split:


Target_State       Bearish  Bullish
Ticker Split                       
AAPL   test             55       67
       train           235      330
       validation       55       66
IBM    test             48       74
       train           269      296
       validation       56       65
MSFT   test             53       69
       train           235      330
       validation       57       64


Final WEEKLY_LSTM_FEATURE_COLS:
['Weekly_Log_Return', 'Weekly_Open_Close_Log_Return', 'Weekly_High_Low_Range', 'Weekly_Volume_Change', 'Rolling_Vol_4', 'Rolling_Vol_12', 'Rolling_Vol_26', 'Momentum_4', 'Momentum_12', 'Momentum_26', 'MA_Gap_4', 'MA_Gap_12', 'MA_Gap_26', 'Drawdown_12', 'Drawdown_26']

Missing values in feature columns:


Price
Weekly_Log_Return               0
Weekly_Open_Close_Log_Return    0
Weekly_High_Low_Range           0
Weekly_Volume_Change            0
Rolling_Vol_4                   0
Rolling_Vol_12                  0
Rolling_Vol_26                  0
Momentum_4                      0
Momentum_12                     0
Momentum_26                     0
MA_Gap_4                        0
MA_Gap_12                       0
MA_Gap_26                       0
Drawdown_12                     0
Drawdown_26                     0
dtype: int64


Sequence shapes by ticker and lookback:
AAPL lookback=4: X_train=(562, 4, 15), y_train=(562,), X_val=(121, 4, 15), y_val=(121,), X_test=(122, 4, 15), y_test=(122,)
AAPL lookback=8: X_train=(558, 8, 15), y_train=(558,), X_val=(121, 8, 15), y_val=(121,), X_test=(122, 8, 15), y_test=(122,)
AAPL lookback=12: X_train=(554, 12, 15), y_train=(554,), X_val=(121, 12, 15), y_val=(121,), X_test=(122, 12, 15), y_test=(122,)
AAPL lookback=26: X_train=(540, 26, 15), y_train=(540,), X_val=(121, 26, 15), y_val=(121,), X_test=(122, 26, 15), y_test=(122,)
MSFT lookback=4: X_train=(562, 4, 15), y_train=(562,), X_val=(121, 4, 15), y_val=(121,), X_test=(122, 4, 15), y_test=(122,)
MSFT lookback=8: X_train=(558, 8, 15), y_train=(558,), X_val=(121, 8, 15), y_val=(121,), X_test=(122, 8, 15), y_test=(122,)
MSFT lookback=12: X_train=(554, 12, 15), y_train=(554,), X_val=(121, 12, 15), y_val=(121,), X_test=(122, 12, 15), y_test=(122,)
MSFT lookback=26: X_train=(540, 26, 15), y_train=(540,), X_val=(121, 26, 15), y

## Summary

This notebook prepares weekly LSTM inputs. The daily OHLCV data is first aggregated to weekly OHLCV data. Weekly features are then constructed, including weekly log return, volume change, rolling volatility, momentum, moving-average gaps, and drawdown.

The prediction target is the next-week market state. A week is labelled bullish if the next-week log return is positive and bearish otherwise. The data is split chronologically using the target date to avoid look-ahead bias.

The notebook prepares sequence arrays for several LSTM lookback windows: 4, 8, 12, and 26 weeks. These windows allow the later training notebook to test whether short-term or longer-term market memory produces better classification performance.

No HMM is trained in this notebook. The next notebook should be `hmm_regime_weekly.ipynb`, which will load `outputs/weekly/model_df_weekly.parquet`, fit a paper-faithful weekly return-only HMM, and save the next-step HMM regime probabilities.
